# Build a Research Agent in 200 Lines

Companion notebook for the [Building a Research Agent: Case Study](https://codemia.io/courses/applied_ai_systems/building_a_research_agent_case_study) lesson on Codemia.

**What you will build**: a research agent that turns a question into search queries, gathers
sources, writes an answer where every sentence cites a source, and then checks each citation
with an entailment test before anything reaches the user. Every step logs what went in, what came
out and how long it took, so you can see exactly where an answer came from.

## Setup

Two optional keys switch the two external pieces to the real thing:

| key | live | without it |
|---|---|---|
| `TAVILY_API_KEY` | Tavily web search | keyword search over 9 bundled snippets |
| `OPENAI_API_KEY` | gpt-4o-mini rewrites queries, writes the answer, checks citations | rule-based stand-ins for all three |

The pipeline, the trace and the citation check are the same code either way. In Colab, add keys
under the key icon (Secrets).

In [ ]:
%pip install -q openai tavily-python

In [ ]:
import json
import math
import os
import re
import time
from collections import Counter

try:
    from google.colab import userdata
    for k in ('OPENAI_API_KEY', 'TAVILY_API_KEY'):
        os.environ.setdefault(k, userdata.get(k) or '')
except Exception:
    pass

USE_LLM = bool(os.environ.get('OPENAI_API_KEY'))
USE_WEB = bool(os.environ.get('TAVILY_API_KEY'))
MODEL = 'gpt-4o-mini'
print('llm:', 'gpt-4o-mini' if USE_LLM else 'stand-in', '| search:', 'tavily' if USE_WEB else 'bundled')

## Concept recap

A research agent is a pipeline more than a free-form agent: **rewrite, search, read, synthesize,
verify**. The interesting engineering is at the two ends.

- **Rewriting** matters because a user's question is a bad search query. "What started the 2003
  blackout and what changed afterwards?" is two questions, and a search engine answers neither
  well while they are glued together.
- **Verification** matters because a model that has just read ten sources will happily cite
  source 3 for a fact it got from source 7, or from nowhere. A citation is a claim, and claims
  get checked.

## The trace

Every step goes through `traced`, which records inputs, outputs and wall time. This is the part
people skip and then regret: when an answer is wrong, the trace tells you whether the search
missed, the synthesis invented, or the checker let it through.

In [ ]:
TRACE = []


def traced(step):
    def wrap(fn):
        def inner(*args, **kwargs):
            t0 = time.perf_counter()
            out = fn(*args, **kwargs)
            TRACE.append({'step': step, 'ms': round((time.perf_counter() - t0) * 1000),
                          'in': args[0] if args else kwargs, 'out': out})
            return out
        return inner
    return wrap


def show_trace(width=100):
    for t in TRACE:
        print(f"--- {t['step']}  ({t['ms']} ms)")
        print('  in :', str(t['in'])[:width])
        out = t['out'] if isinstance(t['out'], list) else [t['out']]
        for o in out[:6]:
            print('  out:', str(o)[:width])

## Search

The bundled corpus is nine short snippets: five about the 2003 Northeast blackout and its
regulatory aftermath, and four near misses (other blackouts, other grids) so that search has
something to get wrong. Offline search is plain BM25, the same scoring a basic search engine uses.

In [ ]:
CORPUS = [
    ('2003 Northeast blackout: what caused it', 'https://example.org/task-force/causes',
     'The 2003 Northeast blackout began on 14 August 2003 and cut power to about 50 million people '
     'in eight US states and Ontario. The joint US-Canada task force found that FirstEnergy in Ohio '
     'lost its control room alarm system that afternoon, so operators did not see the grid degrading.'),
    ('2003 blackout: lines into trees', 'https://example.org/task-force/trees',
     'In the 2003 blackout, three FirstEnergy 345 kV transmission lines failed after sagging into '
     'overgrown trees. With the alarm system down, the line losses went unnoticed.'),
    ('2003 blackout: how the cascade spread', 'https://example.org/task-force/cascade',
     'Once lines in northern Ohio tripped, power flows shifted onto neighbouring lines, which '
     'overloaded and tripped in turn. The cascade spread across the Northeast grid in minutes.'),
    ('After the blackout: the Energy Policy Act of 2005', 'https://example.org/epact-2005',
     'After the 2003 blackout, the Energy Policy Act of 2005 changed US grid regulation by making '
     'electric reliability standards mandatory and enforceable. Before it, the grid reliability '
     'rules were voluntary.'),
    ('NERC becomes the US grid reliability regulator', 'https://example.org/nerc-ero',
     'In 2006 the Federal Energy Regulatory Commission certified NERC as the Electric Reliability '
     'Organization, with authority to enforce grid reliability standards and issue penalties.'),
    ('1965 Northeast blackout', 'https://example.org/1965',
     'The 9 November 1965 Northeast blackout began when a protective relay at the Sir Adam Beck '
     'station in Ontario tripped a transmission line.'),
    ('1977 New York blackout', 'https://example.org/1977',
     'The July 1977 New York City blackout started after lightning strikes on transmission lines.'),
    ('2003 Italy blackout', 'https://example.org/italy-2003',
     'On 28 September 2003 most of Italy lost power in a blackout that started with a tree '
     'flashover on a Swiss line carrying imports into Italy.'),
    ('Grid batteries and frequency regulation', 'https://example.org/batteries',
     'Grid scale batteries respond in milliseconds and are increasingly used for frequency '
     'regulation on power grids.'),
]

WORD = re.compile(r'[a-z0-9]+')
STOP = set('a an the of in on and or to for by was were is are it its that this with what which '
           'did does do how after before from as at be into his her their'.split())


def tokens(text):
    return [w for w in WORD.findall(text.lower()) if w not in STOP]

In [ ]:
DOC_TOKENS = [tokens(t + ' ' + body) for t, _, body in CORPUS]
AVG_LEN = sum(map(len, DOC_TOKENS)) / len(DOC_TOKENS)
DF = Counter(w for doc in DOC_TOKENS for w in set(doc))


def bm25(query, k1=1.5, b=0.75):
    n = len(CORPUS)
    scores = []
    for i, doc in enumerate(DOC_TOKENS):
        tf = Counter(doc)
        s = 0.0
        for w in tokens(query):
            if w in tf:
                idf = math.log(1 + (n - DF[w] + 0.5) / (DF[w] + 0.5))
                s += idf * tf[w] * (k1 + 1) / (tf[w] + k1 * (1 - b + b * len(doc) / AVG_LEN))
        scores.append((s, i))
    return sorted(scores, reverse=True)


@traced('search')
def search(query, k=3):
    if USE_WEB:
        from tavily import TavilyClient
        res = TavilyClient(os.environ['TAVILY_API_KEY']).search(query, max_results=k)
        return [{'title': r['title'], 'url': r['url'], 'text': r['content']} for r in res['results']]
    return [{'title': CORPUS[i][0], 'url': CORPUS[i][1], 'text': CORPUS[i][2]}
            for s, i in bm25(query)[:k] if s > 0]

## The model calls

Three jobs need a model: rewrite the question into queries, write a cited answer, and judge
whether a source supports a sentence. Each has a stand-in so the notebook runs without a key:

- **rewrite**: split the question on "and" into sub-questions.
- **synthesize**: extractive. For each sub-question, take the best matching sentence from the
  sources and cite it. It cannot invent anything, which makes it a useful baseline.
- **entails**: share of the claim's content words found in the source. This is crude on purpose,
  and the Variations section shows exactly how it fails.

In [ ]:
def chat(prompt, json_mode=False):
    from openai import OpenAI
    resp = OpenAI().chat.completions.create(
        model=MODEL, temperature=0, messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'} if json_mode else None)
    return resp.choices[0].message.content


@traced('rewrite')
def rewrite(question):
    if USE_LLM:
        out = chat('Rewrite this research question as 2 to 4 short web search queries, one per '
                   'distinct sub-question. Reply as JSON {"queries": [...]}.\n\n' + question, True)
        return json.loads(out)['queries']
    parts = re.split(r',? and (?=what|how|why|when|who|which)', question.rstrip('?'))
    return [p.strip() + '?' for p in parts if p.strip()]

In [ ]:
SENT = re.compile(r'(?<=[.!?])\s+')


@traced('synthesize')
def synthesize(question, queries, sources):
    if USE_LLM:
        listing = '\n'.join(f'[{i}] {s["title"]}: {s["text"]}' for i, s in enumerate(sources, 1))
        return chat('Answer the question in 2 to 4 sentences using only the sources. End every '
                    'sentence with the number of the source that supports it, like [2]. '
                    f'\n\nQuestion: {question}\n\nSources:\n{listing}')
    out, whole = [], set(tokens(question))
    for q in queries:
        qt = set(tokens(q))
        # Sub-query match counts double; the whole question breaks ties toward on-topic text.
        best = max(((2 * len(qt & set(tokens(sent))) + len(whole & set(tokens(sent))), sent, i)
                    for i, s in enumerate(sources, 1) for sent in SENT.split(s['text'])),
                   key=lambda x: x[0])
        out.append(f'{best[1].rstrip(".")} [{best[2]}].')
    return ' '.join(out)

In [ ]:
@traced('entails')
def entails(claim, source_text, threshold=0.6):
    if USE_LLM:
        out = chat('Does the SOURCE fully support the CLAIM? Unsupported details, wrong numbers or '
                   'reversed meaning mean no. Reply as JSON {"supported": true|false, "why": "..."}.'
                   f'\n\nCLAIM: {claim}\n\nSOURCE: {source_text}', True)
        return json.loads(out)
    claim_words = set(tokens(claim))
    found = claim_words & set(tokens(source_text))
    share = len(found) / max(1, len(claim_words))
    return {'supported': share >= threshold, 'why': f'{share:.0%} of claim words in source'}

## The pipeline

`research()` is the whole agent. Sources are deduplicated by URL and numbered once, so every
citation in the answer points at one entry in one list. Then each sentence is checked against
the source it cites. Sentences that cite nothing, cite a number that does not exist, or fail the
entailment check are flagged, and the answer reports how many claims survived.

In [ ]:
CITE = re.compile(r'\[(\d+)\]')


def research(question, k=3):
    TRACE.clear()
    queries = rewrite(question)
    sources, seen = [], set()
    for q in queries:
        for hit in search(q, k=k):
            if hit['url'] not in seen:
                seen.add(hit['url'])
                sources.append(hit)
    answer = synthesize(question, queries, sources)
    return {'question': question, 'queries': queries, 'sources': sources, 'answer': answer,
            'checks': check_citations(answer, sources)}

In [ ]:
def check_citations(answer, sources):
    checks = []
    for sent in SENT.split(answer.strip()):
        nums = [int(n) for n in CITE.findall(sent)]
        claim = CITE.sub('', sent).strip()
        if not nums:
            checks.append((claim, None, {'supported': False, 'why': 'no citation'}))
            continue
        for n in nums:
            if not 1 <= n <= len(sources):
                checks.append((claim, n, {'supported': False, 'why': f'source [{n}] does not exist'}))
            else:
                checks.append((claim, n, entails(claim, sources[n - 1]['text'])))
    return checks


def report(r):
    print('Q:', r['question'])
    print('queries:', r['queries'])
    for i, s in enumerate(r['sources'], 1):
        print(f'  [{i}] {s["title"]}  {s["url"]}')
    print('\nA:', r['answer'], '\n')
    for claim, n, v in r['checks']:
        mark = 'OK  ' if v['supported'] else 'FLAG'
        print(f'{mark} [{n}] {claim[:70]:<70}  {v["why"][:60]}')
    ok = sum(v['supported'] for _, _, v in r['checks'])
    print(f'\n{ok}/{len(r["checks"])} citations verified')

## Experiment

One question with two halves: what started the blackout, and what changed in regulation because
of it. A good run finds the task force snippets for the first half and the 2005 Act or NERC for the
second, and does not cite the 1965 or Italian blackouts for either.

In [ ]:
r = research('What started the 2003 Northeast blackout, and what changed in US grid regulation '
             'because of it?')
report(r)

In [ ]:
show_trace()

## Variations

Each variation removes or attacks one part of the pipeline.

In [ ]:
# 1. No query rewriting. The glued-together question goes to search as one query. Count what
# each approach found for each half of the question, and how much came along for the ride.
CAUSE, RULES = {'causes', 'trees', 'cascade'}, {'epact-2005', 'nerc-ero'}

def coverage(sources):
    names = {s['url'].rsplit('/', 1)[-1] for s in sources}
    return {'cause': len(names & CAUSE), 'regulation': len(names & RULES),
            'off-topic': len(names - CAUSE - RULES)}

q = 'What started the 2003 Northeast blackout, and what changed in US grid regulation because of it?'
print('one raw query:', coverage(search(q, k=3)))
print('rewritten    :', coverage([h for sub in rewrite(q) for h in search(sub, k=3)]))

In [ ]:
# 2. A miscited sentence. Take the verified answer and point its first citation at the 1965
# snippet instead. The claim is still true; the citation is not. The checker should flag it.
sources = r['sources'] + [{'title': '1965 Northeast blackout', 'url': 'https://example.org/1965',
                           'text': CORPUS[5][2]}]
first = SENT.split(r['answer'].strip())[0]
bad = CITE.sub(f'[{len(sources)}]', first)
for claim, n, v in check_citations(bad, sources):
    print('FLAG' if not v['supported'] else 'OK  ', f'[{n}]', claim[:70], '|', v['why'])

In [ ]:
# 3. Where the word-overlap stand-in breaks. A claim that reverses the source's meaning keeps
# almost every word. The rule-based checker passes it; an LLM judge should not.
src = CORPUS[3][2]
for claim in ['The Energy Policy Act of 2005 made reliability standards mandatory.',
              'The Energy Policy Act of 2005 made reliability standards voluntary.',
              'Grid reliability rules were mandatory before the Energy Policy Act of 2005.']:
    v = entails(claim, src)
    print('OK  ' if v['supported'] else 'FLAG', claim, '|', v['why'][:70])

Things worth noticing:

1. **Decomposition buys recall, not precision.** Each sub-question gets its own top three
   results, so both halves of the question are covered. It also pulls in more off-topic sources
   (other blackouts share most of the words), which is why synthesis and verification downstream
   have to be picky about what they use.
2. **Verification runs per citation, not per answer.** One wrong citation does not sink a good
   answer; it gets flagged on its own, which is what a reviewer needs.
3. **A cheap checker is a filter, not a judge.** Word overlap catches citations to the wrong
   document (variation 2) but not reversed meaning (variation 3). With an `OPENAI_API_KEY` set,
   re-run variation 3 and compare. Production systems use a small entailment model or an LLM
   judge for exactly this reason.
4. **Supported is not the same as relevant.** A true sentence about the Italian blackout, cited
   to the Italian snippet, passes every check here. Before the stand-in scored sentences against
   the whole question it did exactly that. Relevance needs its own check, against the question
   rather than the source.

## What to try next

- Feed the flagged sentences back to `synthesize` with the checker's reasons and ask for a
  rewrite. That turns the pipeline into a loop, and you will need a budget for it.
- Add a `fetch(url)` step that reads the full page for the top source instead of the search
  snippet. Snippets are short, so a true claim can fail verification just because the
  supporting sentence was cut off.
- Log token counts per step from the OpenAI responses and work out the cost per answer. Query
  rewriting is cheap; checking every citation with an LLM usually is not.